# Module 06 — Feature Pyramid Networks

Multi-scale feature extraction is fundamental to detecting objects at all sizes.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import matplotlib.pyplot as plt
from fpn import FPN, PANet, BiFPN
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Building an FPN

We attach FPN to a ResNet backbone.

In [ ]:
backbone = models.resnet50(weights=None)
x = torch.randn(1, 3, 512, 512)

# Extract intermediate feature maps
with torch.no_grad():
    c2 = backbone.layer1(backbone.relu(backbone.bn1(backbone.conv1(backbone.maxpool(x if False else backbone.relu(backbone.bn1(backbone.conv1(x))))))))

# Simpler: hook-based extraction
feature_maps = {}
for name in ['layer1','layer2','layer3','layer4']:
    getattr(backbone, name).register_forward_hook(
        lambda m, i, o, n=name: feature_maps.__setitem__(n, o)
    )
bbackbone = backbone.eval()
with torch.no_grad(): backbone(x)

print('Backbone feature map shapes:')
for name, feat in feature_maps.items():
    print(f'  {name}: {tuple(feat.shape)}')

# Channel counts: [256, 512, 1024, 2048]
in_channels = [feature_maps[k].shape[1] for k in ['layer1','layer2','layer3','layer4']]
fpn = FPN(in_channels, out_channels=256)

feat_dict = {str(i): feature_maps[k] for i, k in enumerate(['layer1','layer2','layer3','layer4'])}
with torch.no_grad():
    pyramid = fpn(feat_dict)

print('\nFPN output shapes:')
for k, v in pyramid.items():
    print(f'  {k}: {tuple(v.shape)}')

## 2. Level Assignment

Objects are assigned to pyramid levels based on their area.

In [ ]:
import math
def assign_level(box_area, k0=4, min_level=2, max_level=5):
    """FPN level assignment rule."""
    return int(math.floor(k0 + math.log2(math.sqrt(box_area) / 224)))

areas = [32**2, 64**2, 128**2, 256**2, 512**2]
for area in areas:
    level = max(2, min(5, assign_level(area)))
    side = math.sqrt(area)
    print(f'Object side={side:.0f}px, area={area}: → P{level}')

## Exercise — Visualise FPN Feature Maps

Using the code above, load any image and visualise the FPN feature maps
(P2-P5) by plotting the mean activation across channels for each level.

In [ ]:
### EXERCISE
# 1. Load an image and compute FPN feature maps
# 2. Plot mean(abs(feat), axis=channels) for each level
# Hint: feat.abs().mean(dim=1).squeeze() gives a 2D activation map

# TODO: implement visualization